# Multi-Armed Bandits — companion notebook

Runnable companion for [`multi-armed-bandits.md`](./multi-armed-bandits.md).

Simulate a 10-arm Bernoulli bandit under three policies — $\varepsilon$-greedy, UCB1, Thompson sampling — and compare cumulative regret averaged over many seeds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. Environment

Ten Bernoulli arms with means drawn uniformly in $[0.1, 0.9]$ but FIXED across seeds (so cross-policy comparison is fair). Horizon $T = 2000$, averaged over 200 seeds.

In [ ]:
K = 10
T = 2000
n_seeds = 200

true_means = rng.uniform(0.1, 0.9, size=K)
best_mean = true_means.max()
print('arm means:', true_means)
print(f'best arm  : index {true_means.argmax()} with mean {best_mean:.3f}')

In [ ]:
def run_eps_greedy(true_means, T, seed, eps=0.1):
    local = np.random.default_rng(seed)
    K = len(true_means)
    counts = np.zeros(K); sums = np.zeros(K)
    rewards = np.empty(T)
    for t in range(T):
        if local.random() < eps or counts.min() == 0:
            a = int(local.integers(K))
        else:
            a = int(np.argmax(sums / np.maximum(counts, 1)))
        r = float(local.random() < true_means[a])
        counts[a] += 1; sums[a] += r
        rewards[t] = r
    return rewards

def run_ucb1(true_means, T, seed):
    local = np.random.default_rng(seed)
    K = len(true_means)
    counts = np.zeros(K); sums = np.zeros(K)
    rewards = np.empty(T)
    for t in range(T):
        if t < K:
            a = t  # one pull per arm to start
        else:
            mean_hat = sums / counts
            bonus = np.sqrt(2 * np.log(t + 1) / counts)
            a = int(np.argmax(mean_hat + bonus))
        r = float(local.random() < true_means[a])
        counts[a] += 1; sums[a] += r
        rewards[t] = r
    return rewards

def run_thompson(true_means, T, seed, alpha0=1.0, beta0=1.0):
    local = np.random.default_rng(seed)
    K = len(true_means)
    alpha = np.full(K, alpha0); beta = np.full(K, beta0)
    rewards = np.empty(T)
    for t in range(T):
        samples = local.beta(alpha, beta)
        a = int(np.argmax(samples))
        r = float(local.random() < true_means[a])
        alpha[a] += r; beta[a] += 1 - r
        rewards[t] = r
    return rewards

In [ ]:
policies = {
    'eps-greedy (0.1)': lambda s: run_eps_greedy(true_means, T, s, eps=0.1),
    'UCB1'            : lambda s: run_ucb1(true_means, T, s),
    'Thompson (Beta)' : lambda s: run_thompson(true_means, T, s),
}

regret = {name: np.zeros(T) for name in policies}
for s in range(n_seeds):
    for name, fn in policies.items():
        r = fn(1000 + s)
        regret[name] += best_mean - r          # instantaneous regret

for name in regret:
    regret[name] = np.cumsum(regret[name] / n_seeds)

print('Final cumulative regret (lower = better):')
for name, r in regret.items():
    print(f'  {name:20s}  {r[-1]:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ts = np.arange(1, T + 1)
for name, r in regret.items():
    ax.plot(ts, r, label=name, lw=1.5)
# Reference: O(log T) shape for visualization
ax.plot(ts, 8 * np.log(ts), 'k:', alpha=0.5, label=r'$\propto \log T$ (reference shape)')
ax.set_xlabel('round t'); ax.set_ylabel('cumulative pseudo-regret')
ax.set_title(f'Bernoulli bandit, K={K} arms, averaged over {n_seeds} seeds')
ax.set_xscale('log')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

**Reading the plot.**
- $\varepsilon$-greedy grows roughly linearly (constant exploration probability ⇒ $\Theta(\varepsilon T)$).
- UCB1 and Thompson are both $O(\log T)$ and look like flattening lines on the log-x axis.
- Thompson typically edges UCB1 in practice on Bernoulli bandits — the gap is constant-factor.